In [26]:
#from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
#from langchain_openai import ChatOpenAI
from langchain_community.document_loaders.directory import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

db = Chroma.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 5})

In [27]:
len(chunks)

99

In [48]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
import re

query = "Who owns the restaurant?"


QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate five
    different versions of the given user question to retrieve relevant documents from a vector
    database. By generating multiple perspectives on the user question, your goal is to help
    the user overcome some of the limitations of the distance-based similarity search.
    Provide these alternative question like this:
    <<question1>>
    <<question2>>
    Only provide the query, no numbering.
    Original question: {question}""",
)


def split_and_clean_text(input_text):
    return [item.strip() for item in re.split(r"\n", input_text) if item.strip()]

In [49]:
model = chat_model
multiquery_chain = (
    QUERY_PROMPT | model | StrOutputParser() | RunnableLambda(split_and_clean_text)
)

In [50]:
list_of_questions = multiquery_chain.invoke(query)

In [51]:
list_of_questions

['Who is the owner of the restaurant?',
 'Who holds ownership of the restaurant?',
 "Who is responsible for the restaurant's management?",
 'Which individual or entity owns the restaurant?',
 'Who is the proprietor of the restaurant?']

In [52]:
for j in list_of_questions:
    print(j)

Who is the owner of the restaurant?
Who holds ownership of the restaurant?
Who is responsible for the restaurant's management?
Which individual or entity owns the restaurant?
Who is the proprietor of the restaurant?


In [53]:
list_of_questions[0]

'Who is the owner of the restaurant?'

In [54]:
retriever.invoke("Who is the owner of the restaurant?")

[Document(metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,')]

In [55]:
docs = []
for q in list_of_questions:
    docs.append(retriever.invoke(q))

In [56]:
len(docs)

5

In [57]:
for j in docs:
    print(len(j))

5
5
5
5
5


In [58]:
def flatten_and_unique_documents(documents):
    flattened_docs = [doc for sublist in documents for doc in sublist]

    unique_docs = []
    unique_contents = set()
    for doc in flattened_docs:
        if doc.page_content not in unique_contents:
            unique_docs.append(doc)
            unique_contents.add(doc.page_content)

    return unique_docs

In [59]:
flatten_and_unique_documents(documents=docs)

[Document(metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
 Document(metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico")]

In [60]:
HYDE_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate five hypothetical answers to the user's query. These answers should offer diverse perspectives or interpretations, aiding in a comprehensive understanding of the query. Present the hypothetical answers as follows:

    <<Answer considering a specific perspective>>
    <<Answer from a different angle>>
    <<Answer exploring an alternative possibility>>
    <<Answer providing a contrasting viewpoint>>
    <<Answer that includes a unique insight>>

    Note: Present only the hypothetical answers, without numbering (or "-", "1.", "*") and so on, to provide a range of potential interpretations or solutions related to the query.
    Original question: {question}""",
)

In [61]:
hyde_chain = (
    HYDE_PROMPT | model | StrOutputParser() | RunnableLambda(split_and_clean_text)
)

In [62]:
list_of_questions = hyde_chain.invoke("Who is the owner of the restaurant")
list_of_questions

['The owner of the restaurant is likely the individual or entity who holds the legal title and primary control over its operations and assets.',
 'From a different angle, the restaurant might be owned collectively by a partnership or a corporation rather than a single person, implying shared ownership and responsibilities.',
 'An alternative possibility is that the restaurant is leased or rented from a property owner, meaning the person running it might not be the actual owner but simply a tenant or franchisee.',
 'Contrasting viewpoint, in some cases, the "owner" could be a franchise organization or brand that oversees multiple locations, making the local manager or operator a representative rather than the true owner.',
 'A unique insight might be that the owner could be a silent investor or a family trust, where the day-to-day management is handled by professionals, and the ownership is maintained behind the scenes.']

In [63]:
docs = [retriever.invoke(q) for q in list_of_questions]
flatten_and_unique_documents(documents=docs)

[Document(metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
 Document(metadata={'source': 'data\\founder.txt'}, page_content='cuisine, reflected Amico’s journey and his commitment to excellence. Patrons were not just diners; they were part of an'),
 Document(metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico")]